# Local Experts Pipeline — 100 Independent Pixel Models

Predicts the **t-1** state of every cell on a 10×10 GoL board by training one binary classifier per pixel.

**Phase 1** — train & save 100 models (`pixel_models/model_pixel_i.keras`).  
**Phase 2** — load all models, reconstruct full boards, evaluate comprehensively.

---
## 0 · Imports & Configuration

In [ ]:
import importlib
import functions
importlib.reload(functions)
from functions import *

import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt

print(f'TensorFlow {tf.__version__}')

In [ ]:
# ─── Global hyper-parameters ──────────────────────────────────────────────────
SIZE           = 10
AMOUNT_BOARDS  = 1000
GEN            = 4        # number of generations in the reverse chain
N_PIXELS       = SIZE * SIZE   # 100

TEST_SIZE      = 0.10
VAL_SIZE       = 0.10
RANDOM_STATE   = 365

EPOCHS         = 50       # upper bound — early stopping will cut this short
BATCH_SIZE     = 256
ES_PATIENCE    = 5        # early-stopping patience

# ── Architecture selector ─────────────────────────────────────────────────────
# Choose ONE: 'cnn' | 'rnn' | 'crnn' | 'rcnn'
# Each produces its own model subdirectory so runs never collide.
ACTIVE_ARCH    = 'cnn'

# Directory encodes dataset config + architecture
MODEL_DIR      = f'pixel_models/size{SIZE}_gen{GEN}/{ACTIVE_ARCH}'
os.makedirs(MODEL_DIR, exist_ok=True)
print(f'Active architecture : {ACTIVE_ARCH.upper()}')
print(f'Model directory     : {MODEL_DIR}/')

---
## 1 · Load Raw Data Once

Features **X** are identical for every pixel — only the target column **y** changes.  
Load the raw DataFrame here so we avoid re-reading from disk 100 times.

In [ ]:
reverse_df = load_reverse_df(SIZE, AMOUNT_BOARDS, GEN)
print(f'Dataset shape: {reverse_df.shape}')

# ── Build the shared feature matrix (all gen-1 boards, flat) ──────────────────
amount_features = (GEN - 1) * SIZE * SIZE      # columns used as input
X_raw = reverse_df.iloc[:, :amount_features].astype(np.int8)

# ── Shared train / val / test INDEX split (same partitioning for every pixel) ─
idx = np.arange(len(X_raw))
idx_trainval, idx_test = train_test_split(idx, test_size=TEST_SIZE,  random_state=RANDOM_STATE)
idx_train,    idx_val  = train_test_split(idx_trainval, test_size=VAL_SIZE, random_state=RANDOM_STATE)

# 4-D numpy arrays for CNN  (n, SIZE, SIZE, gen-1)
X_train_4d = X_raw.iloc[idx_train].to_numpy().reshape(-1, SIZE, SIZE, GEN-1).astype(np.float32)
X_val_4d   = X_raw.iloc[idx_val  ].to_numpy().reshape(-1, SIZE, SIZE, GEN-1).astype(np.float32)
X_test_4d  = X_raw.iloc[idx_test ].to_numpy().reshape(-1, SIZE, SIZE, GEN-1).astype(np.float32)

print(f'Train: {X_train_4d.shape} | Val: {X_val_4d.shape} | Test: {X_test_4d.shape}')

# ── Full ground-truth boards at t-1  (n, 100) — used in Phase 2 ──────────────
target_cols = [f'Col_{amount_features + i}' for i in range(N_PIXELS)]
y_full = reverse_df[target_cols].astype(np.int8).to_numpy()  # (N, 100)

y_test_full = y_full[idx_test]      # (N_test, 100)  ← Phase 2 ground truth
print(f'y_test_full shape: {y_test_full.shape}')

---
## 2 · Architecture Registry

Four architectures are registered below. Each entry has:
- `build_fn`   — builds & compiles a fresh Keras model
- `reshape_fn` — reshapes `X` from `(n, SIZE, SIZE, GEN-1)` to the model's expected input

| Key    | Architecture                         | Input shape                        |
|--------|--------------------------------------|------------------------------------|
| `cnn`  | Lightweight CNN (2× Conv2D)          | `(SIZE, SIZE, GEN-1)`              |
| `rnn`  | Single LSTM                          | `(GEN-1, SIZE×SIZE)`               |
| `crnn` | TimeDistributed Conv2D + BiLSTM      | `(GEN-1, SIZE, SIZE, 1)`           |
| `rcnn` | ConvLSTM2D                           | `(GEN-1, SIZE, SIZE, 1)`           |

**To swap**: change `ACTIVE_ARCH` in the config cell and re-run. Everything else is automatic.

In [ ]:
TIMESTEPS = GEN - 1   # number of board-state channels = temporal steps

# ══════════════════════════════════════════════════════════════════════════════
# Reshape helpers — all take (n, SIZE, SIZE, TIMESTEPS) and return the
# tensor the corresponding model expects.
# ══════════════════════════════════════════════════════════════════════════════

def reshape_for_cnn(X: np.ndarray) -> np.ndarray:
    """(n, SIZE, SIZE, TIMESTEPS) — unchanged, channels = generations."""
    return X.astype(np.float32)

def reshape_for_rnn(X: np.ndarray) -> np.ndarray:
    """(n, TIMESTEPS, SIZE*SIZE) — each board flattened, sequence over gens."""
    n = X.shape[0]
    return X.transpose(0, 3, 1, 2).reshape(n, TIMESTEPS, SIZE * SIZE).astype(np.float32)

def reshape_for_crnn(X: np.ndarray) -> np.ndarray:
    """(n, TIMESTEPS, SIZE, SIZE, 1) — spatial dims kept, channel dim added."""
    n = X.shape[0]
    return X.transpose(0, 3, 1, 2).reshape(n, TIMESTEPS, SIZE, SIZE, 1).astype(np.float32)

def reshape_for_rcnn(X: np.ndarray) -> np.ndarray:
    """(n, TIMESTEPS, SIZE, SIZE, 1) — same layout as crnn (ConvLSTM2D)."""
    return reshape_for_crnn(X)


# ══════════════════════════════════════════════════════════════════════════════
# Model builders — lightweight versions (fast enough to train 100×)
# ══════════════════════════════════════════════════════════════════════════════

def build_cnn_pixel(input_shape: tuple) -> tf.keras.Model:
    """Two Conv2D blocks → Dense head.  ~7 k params."""
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1,  activation='sigmoid'),
    ], name='pixel_cnn')


def build_rnn_pixel(input_shape: tuple) -> tf.keras.Model:
    """Single LSTM treating each board generation as one timestep.  ~17 k params."""
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),          # (TIMESTEPS, SIZE*SIZE)
        tf.keras.layers.LSTM(32, activation='tanh'),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(1,  activation='sigmoid'),
    ], name='pixel_rnn')


def build_crnn_pixel(input_shape: tuple) -> tf.keras.Model:
    """TimeDistributed Conv2D (spatial) + BiLSTM (temporal).  ~24 k params.
    Mirrors build_and_train_crnn_v3 from functions.py (lightweight edition)."""
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),          # (TIMESTEPS, SIZE, SIZE, 1)

        # Spatial feature extraction — same Conv applied at every timestep
        tf.keras.layers.TimeDistributed(
            tf.keras.layers.Conv2D(8, (3, 3), activation='relu', padding='same')
        ),
        tf.keras.layers.TimeDistributed(tf.keras.layers.MaxPooling2D((2, 2))),
        tf.keras.layers.TimeDistributed(tf.keras.layers.Flatten()),

        # Temporal reasoning
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32, activation='tanh')),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(1,  activation='sigmoid'),
    ], name='pixel_crnn')


def build_rcnn_pixel(input_shape: tuple) -> tf.keras.Model:
    """ConvLSTM2D — learns spatial AND temporal patterns jointly.  ~20 k params.
    Mirrors build_and_train_rcnn from functions.py (lightweight edition)."""
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),          # (TIMESTEPS, SIZE, SIZE, 1)

        tf.keras.layers.ConvLSTM2D(
            filters=16, kernel_size=(3, 3),
            activation='relu', padding='same',
            return_sequences=False,
        ),
        tf.keras.layers.BatchNormalization(),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1,  activation='sigmoid'),
    ], name='pixel_rcnn')


# ══════════════════════════════════════════════════════════════════════════════
# Registry — maps arch key → build + reshape functions + input shape
# ══════════════════════════════════════════════════════════════════════════════

ARCH_REGISTRY = {
    'cnn': {
        'build_fn':    build_cnn_pixel,
        'reshape_fn':  reshape_for_cnn,
        'input_shape': (SIZE, SIZE, TIMESTEPS),
    },
    'rnn': {
        'build_fn':    build_rnn_pixel,
        'reshape_fn':  reshape_for_rnn,
        'input_shape': (TIMESTEPS, SIZE * SIZE),
    },
    'crnn': {
        'build_fn':    build_crnn_pixel,
        'reshape_fn':  reshape_for_crnn,
        'input_shape': (TIMESTEPS, SIZE, SIZE, 1),
    },
    'rcnn': {
        'build_fn':    build_rcnn_pixel,
        'reshape_fn':  reshape_for_rcnn,
        'input_shape': (TIMESTEPS, SIZE, SIZE, 1),
    },
}

# ── Validate the active arch and prepare reshaped train/val arrays ────────────
assert ACTIVE_ARCH in ARCH_REGISTRY, f"Unknown arch '{ACTIVE_ARCH}'. Choose: {list(ARCH_REGISTRY)}"
arch_cfg    = ARCH_REGISTRY[ACTIVE_ARCH]
reshape_fn  = arch_cfg['reshape_fn']
build_fn    = arch_cfg['build_fn']
INPUT_SHAPE = arch_cfg['input_shape']

X_train_arch = reshape_fn(X_train_4d)
X_val_arch   = reshape_fn(X_val_4d)
X_test_arch  = reshape_fn(X_test_4d)

print(f'Architecture  : {ACTIVE_ARCH.upper()}')
print(f'Input shape   : {INPUT_SHAPE}')
print(f'X_train_arch  : {X_train_arch.shape}')

# Param count
_demo = build_fn(INPUT_SHAPE)
_demo.compile(optimizer='adam', loss='binary_crossentropy')
print(f'Params/model  : {_demo.count_params():,}')
_demo.summary()
del _demo

---
## PHASE 1 — Train All Architectures (100 models each)

In [ ]:
def get_callbacks(patience: int = ES_PATIENCE) -> list:
    """Standard callbacks: early stopping + learning-rate reduction."""
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=patience,
            restore_best_weights=True,
            verbose=0,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=max(1, patience // 2),
            min_lr=1e-5,
            verbose=0,
        ),
    ]


def compute_class_weights(y: np.ndarray) -> dict:
    """Balanced class weights for skewed binary targets."""
    y_flat = y.reshape(-1).astype(int)
    total  = len(y_flat)
    n0, n1 = int((y_flat == 0).sum()), int((y_flat == 1).sum())
    if n0 == 0 or n1 == 0:
        return None
    return {0: total / (2.0 * n0), 1: total / (2.0 * n1)}

In [ ]:
# ─── Train all architectures — 100 pixel models each ─────────────────────────
# Skips any pixel that already has a saved model (safe to re-run / resume).

grand_log = {}   # arch_key → list of per-pixel dicts

for arch_key, arch_cfg in ARCH_REGISTRY.items():

    arch_dir = f'pixel_models/size{SIZE}_gen{GEN}/{arch_key}'
    os.makedirs(arch_dir, exist_ok=True)

    # Pre-reshape X once per architecture
    X_tr = arch_cfg['reshape_fn'](X_train_4d)
    X_vl = arch_cfg['reshape_fn'](X_val_4d)
    input_shape = arch_cfg['input_shape']

    print(f'\n{"═"*54}')
    print(f'  Training {arch_key.upper()} — input {input_shape}')
    print(f'{"═"*54}')

    arch_log = []
    t0_arch  = time.time()

    for pixel_idx in range(N_PIXELS):
        save_path = os.path.join(
            arch_dir,
            f'model_pixel{pixel_idx:03d}_size{SIZE}_gen{GEN}_{arch_key}.keras'
        )

        # Resume support — skip pixels already trained
        if os.path.exists(save_path):
            if pixel_idx % 10 == 0:
                print(f'  [pixel {pixel_idx:03d}] already exists, skipping ...')
            continue

        t0_pixel = time.time()

        # Target
        target_col = f'Col_{amount_features + pixel_idx}'
        y_all   = reverse_df[target_col].astype(np.float32).to_numpy()
        y_train = y_all[idx_train].reshape(-1, 1)
        y_val   = y_all[idx_val  ].reshape(-1, 1)

        # Build & train
        model = arch_cfg['build_fn'](input_shape)
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

        history = model.fit(
            X_tr, y_train,
            validation_data=(X_vl, y_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=get_callbacks(),
            class_weight=compute_class_weights(y_train),
            verbose=0,
        )

        model.save(save_path)

        best_val_acc  = max(history.history['val_accuracy'])
        best_val_loss = min(history.history['val_loss'])
        epochs_run    = len(history.history['loss'])
        elapsed       = time.time() - t0_pixel

        arch_log.append({
            'pixel_idx':     pixel_idx,
            'epochs_run':    epochs_run,
            'best_val_acc':  best_val_acc,
            'best_val_loss': best_val_loss,
            'time_s':        elapsed,
        })

        if pixel_idx % 10 == 0 or pixel_idx == N_PIXELS - 1:
            print(f'  [{arch_key.upper()}] pixel {pixel_idx+1:3d}/100  '
                  f'val_acc={best_val_acc:.3f}  '
                  f'epochs={epochs_run:2d}  '
                  f'arch_total={time.time()-t0_arch:.0f}s')

        tf.keras.backend.clear_session()

    grand_log[arch_key] = arch_log
    trained = len(arch_log)
    skipped = N_PIXELS - trained
    print(f'\n  ✓ {arch_key.upper()} done — '
          f'{trained} trained, {skipped} skipped (already existed)')

print(f'\n{"═"*54}')
print('✓ All architectures complete.')

# Summary table across architectures
summary_rows = []
for arch_key, log in grand_log.items():
    if log:
        accs = [r['best_val_acc'] for r in log]
        summary_rows.append({
            'Architecture': arch_key.upper(),
            'Pixels trained': len(log),
            'Mean val_acc':   f'{np.mean(accs):.3f}',
            'Min val_acc':    f'{np.min(accs):.3f}',
            'Max val_acc':    f'{np.max(accs):.3f}',
        })
if summary_rows:
    display(pd.DataFrame(summary_rows).set_index('Architecture'))

In [ ]:
# ─── Per-architecture training plots ─────────────────────────────────────────
archs_with_data = {k: v for k, v in grand_log.items() if v}

if archs_with_data:
    fig, axes = plt.subplots(2, len(archs_with_data),
                             figsize=(5 * len(archs_with_data), 7),
                             squeeze=False)

    for col, (arch_key, log) in enumerate(archs_with_data.items()):
        log_df = pd.DataFrame(log)

        # Val accuracy per pixel
        axes[0, col].bar(log_df['pixel_idx'], log_df['best_val_acc'], width=1.0)
        axes[0, col].axhline(log_df['best_val_acc'].mean(), color='red',
                             linestyle='--',
                             label=f"mean={log_df['best_val_acc'].mean():.3f}")
        axes[0, col].set_title(f'{arch_key.upper()} — Val Accuracy')
        axes[0, col].set_xlabel('Pixel index')
        axes[0, col].set_ylabel('Accuracy')
        axes[0, col].legend(fontsize=8)
        axes[0, col].set_ylim(0.5, 1.0)

        # Epochs per pixel
        axes[1, col].bar(log_df['pixel_idx'], log_df['epochs_run'],
                         width=1.0, color='orange')
        axes[1, col].set_title(f'{arch_key.upper()} — Early-Stop Epochs')
        axes[1, col].set_xlabel('Pixel index')
        axes[1, col].set_ylabel('Epochs')

    plt.tight_layout()
    plt.show()

---
## PHASE 2 — Full Board Reconstruction & Evaluation

> **Architecture swap point**: if you trained a non-Keras model (e.g. Random Forest),
> replace `predict_all_pixels_keras()` with your own function that returns an
> `(N_test, 100)` int array of 0/1 predictions, then call `evaluate_full_boards()`
> exactly as shown below.

In [ ]:
# ─── 2a. Discover & load all 100 saved models ────────────────────────────────
# ACTIVE_ARCH controls which set of models is loaded — change it in cell-3.
import glob as _glob

expected_pattern = os.path.join(
    MODEL_DIR, f'model_pixel*_size{SIZE}_gen{GEN}_{ACTIVE_ARCH}.keras'
)
found_paths = sorted(_glob.glob(expected_pattern))

print(f'Architecture : {ACTIVE_ARCH.upper()}')
print(f'Looking in   : {MODEL_DIR}/')
print(f'Found        : {len(found_paths)} / {N_PIXELS} model files')

if len(found_paths) != N_PIXELS:
    missing = [
        i for i in range(N_PIXELS)
        if not os.path.exists(
            os.path.join(MODEL_DIR, f'model_pixel{i:03d}_size{SIZE}_gen{GEN}_{ACTIVE_ARCH}.keras')
        )
    ]
    print(f'  ⚠ Missing pixel indices: {missing}')
else:
    print('  ✓ All 100 models present')

pixel_models = []
for i in range(N_PIXELS):
    path = os.path.join(MODEL_DIR, f'model_pixel{i:03d}_size{SIZE}_gen{GEN}_{ACTIVE_ARCH}.keras')
    pixel_models.append(tf.keras.models.load_model(path))

print(f'\nLoaded {len(pixel_models)} models.')
print(f'  Model name   : {pixel_models[0].name}')
print(f'  Input shape  : {pixel_models[0].input_shape}')
print(f'  Params each  : {pixel_models[0].count_params():,}')

In [ ]:
# ─── 2b. Predict all pixels & reconstruct boards ─────────────────────────────
# X_test_arch is already reshaped for ACTIVE_ARCH (computed in cell-7).

def predict_all_pixels_keras(
    models: list,
    X_test: np.ndarray,
    batch_size: int = 512,
    threshold: float = 0.5,
) -> np.ndarray:
    """
    Run each Keras pixel-model over X_test and stack predictions.

    Parameters
    ----------
    models    : list of 100 compiled Keras models
    X_test    : pre-reshaped array matching the model's input shape
    batch_size: inference batch size
    threshold : sigmoid threshold for class 1

    Returns
    -------
    pred_boards : (N, 100) int8 array — reconstructed t-1 boards
    """
    n_samples = X_test.shape[0]
    pred_boards = np.zeros((n_samples, len(models)), dtype=np.int8)

    for i, model in enumerate(models):
        probs = model.predict(X_test, batch_size=batch_size, verbose=0).flatten()
        pred_boards[:, i] = (probs >= threshold).astype(np.int8)
        if (i + 1) % 10 == 0 or i == len(models) - 1:
            print(f'  Predicted pixel {i+1:3d}/100', end='\r')

    print()
    return pred_boards


print(f'Running inference with {ACTIVE_ARCH.upper()} models...')
t0 = time.time()
pred_boards = predict_all_pixels_keras(pixel_models, X_test_arch)
print(f'Inference done in {time.time()-t0:.1f}s')
print(f'pred_boards shape: {pred_boards.shape}')   # (N_test, 100)

In [ ]:
# ─── 2c. Comprehensive metric suite ──────────────────────────────────────────

def evaluate_full_boards(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    architecture_name: str = '',
) -> dict:
    """
    Compute and display full-board reconstruction metrics.

    Parameters
    ----------
    y_true            : (N, 100) int  — ground-truth t-1 boards
    y_pred            : (N, 100) int  — predicted t-1 boards
    architecture_name : label shown in the report

    Returns
    -------
    metrics : dict — all computed values, for easy comparison later
    """
    assert y_true.shape == y_pred.shape, 'Shape mismatch'
    n_samples, _ = y_true.shape

    # 1. Average pixel-wise accuracy
    pixel_accs  = (y_true == y_pred).mean(axis=0)   # (100,)
    avg_pix_acc = pixel_accs.mean()

    # 2. Exact board match accuracy
    exact_matches   = (y_true == y_pred).all(axis=1)
    exact_board_acc = exact_matches.mean()
    n_exact         = int(exact_matches.sum())

    # 3. Precision / Recall / F1 for the Alive class (flattened)
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()
    precision = precision_score(y_true_flat, y_pred_flat, zero_division=0)
    recall    = recall_score   (y_true_flat, y_pred_flat, zero_division=0)
    f1        = f1_score       (y_true_flat, y_pred_flat, zero_division=0)

    # 4. Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_true_flat, y_pred_flat).ravel()

    sep = '─' * 52
    print(f'\n{sep}')
    print(f'  Architecture : {architecture_name}')
    print(f'  Test samples : {n_samples:,}')
    print(sep)
    print(f'  1. Avg Pixel-wise Accuracy    : {avg_pix_acc:.4f}  ({avg_pix_acc*100:.2f}%)')
    print(f'  2. Exact Board Match Accuracy : {exact_board_acc:.4f}  ({n_exact}/{n_samples} boards)')
    print(f'  3. Alive-class Metrics (flat):')
    print(f'       Precision : {precision:.4f}')
    print(f'       Recall    : {recall:.4f}')
    print(f'       F1-score  : {f1:.4f}')
    print(f'  Confusion matrix (flat pixels):')
    print(f'       TP={tp:,}  FP={fp:,}  FN={fn:,}  TN={tn:,}')
    print(sep)

    return dict(
        architecture         = architecture_name,
        n_samples            = n_samples,
        avg_pixel_acc        = float(avg_pix_acc),
        exact_board_acc      = float(exact_board_acc),
        n_exact_matches      = n_exact,
        precision_alive      = float(precision),
        recall_alive         = float(recall),
        f1_alive             = float(f1),
        tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn),
        pixel_accs_per_pixel = pixel_accs.tolist(),
    )


# Run evaluation for the currently active architecture
current_results = evaluate_full_boards(
    y_true=y_test_full,
    y_pred=pred_boards,
    architecture_name=f'{ACTIVE_ARCH.upper()} (Local Experts, size={SIZE}, gen={GEN})',
)

In [ ]:
# ─── 2d. Per-pixel accuracy heatmap ──────────────────────────────────────────
pix_accs_grid = np.array(results_cnn['pixel_accs_per_pixel']).reshape(SIZE, SIZE)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(pix_accs_grid, vmin=0.5, vmax=1.0, cmap='RdYlGn')
plt.colorbar(im, ax=ax, label='Per-pixel accuracy')
ax.set_title('Per-pixel Test Accuracy — 10×10 Board')
ax.set_xlabel('Column')
ax.set_ylabel('Row')
for r in range(SIZE):
    for c in range(SIZE):
        ax.text(c, r, f'{pix_accs_grid[r,c]:.2f}', ha='center', va='center',
                fontsize=6, color='black')
plt.tight_layout()
plt.show()

In [ ]:
# ─── 2e. Sample reconstructions ──────────────────────────────────────────────
# Visualise a few test boards: ground truth vs prediction

N_SHOW = 5
fig, axes = plt.subplots(N_SHOW, 2, figsize=(4, N_SHOW * 2))

for row in range(N_SHOW):
    gt   = y_test_full[row].reshape(SIZE, SIZE)
    pred = pred_boards[row].reshape(SIZE, SIZE)
    match = 'EXACT ✓' if (gt == pred).all() else f'{(gt==pred).mean()*100:.0f}% pixels'

    axes[row, 0].imshow(gt,   cmap='binary', vmin=0, vmax=1)
    axes[row, 0].set_title(f'Sample {row} — Ground Truth', fontsize=8)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(pred, cmap='binary', vmin=0, vmax=1)
    axes[row, 1].set_title(f'Prediction ({match})', fontsize=8)
    axes[row, 1].axis('off')

plt.suptitle('Ground Truth vs Predicted t-1 Boards', y=1.01)
plt.tight_layout()
plt.show()

---
## 3 · Multi-Architecture Comparison Table

After training a second set of models with a different architecture, call
`evaluate_full_boards()` again and append the result dict to `all_results`.
The cell below will always render a clean comparison table.

In [ ]:
# ─── Comparison table ─────────────────────────────────────────────────────────
# After running Phase 1+2 for each architecture, append its result dict here.
# The table is rebuilt every time this cell runs.
#
# Workflow:
#   1. Set ACTIVE_ARCH = 'cnn'  → run all cells → append current_results below
#   2. Set ACTIVE_ARCH = 'rnn'  → run all cells → append current_results below
#   3. Set ACTIVE_ARCH = 'crnn' → run all cells → append current_results below
#   4. Set ACTIVE_ARCH = 'rcnn' → run all cells → append current_results below

all_results = [
    current_results,          # automatically filled from the last run
    # Paste additional result dicts here, e.g.:
    # {'architecture': 'RNN (Local Experts, size=10, gen=4)',  'avg_pixel_acc': 0.82, ...},
]

comparison_df = pd.DataFrame([
    {
        'Architecture':       r['architecture'],
        'Avg Pixel Acc':      f"{r['avg_pixel_acc']:.4f}",
        'Exact Board Acc':    f"{r['exact_board_acc']:.4f}",
        'Exact Boards':       f"{r['n_exact_matches']}/{r['n_samples']}",
        'Precision (Alive)':  f"{r['precision_alive']:.4f}",
        'Recall (Alive)':     f"{r['recall_alive']:.4f}",
        'F1 (Alive)':         f"{r['f1_alive']:.4f}",
    }
    for r in all_results
])

display(comparison_df.set_index('Architecture'))

---
## 4 · Evaluate All Architectures in One Pass

Runs Phase 2 for every architecture whose 100 model files are present on disk.
No need to change `ACTIVE_ARCH` — just run this cell.

In [ ]:
all_arch_results = {}   # arch_key → metrics dict

for arch_key, arch_cfg in ARCH_REGISTRY.items():

    # ── 1. Check all 100 model files exist ────────────────────────────────────
    arch_dir = f'pixel_models/size{SIZE}_gen{GEN}/{arch_key}'
    missing = [
        i for i in range(N_PIXELS)
        if not os.path.exists(
            os.path.join(arch_dir, f'model_pixel{i:03d}_size{SIZE}_gen{GEN}_{arch_key}.keras')
        )
    ]
    if missing:
        print(f'[{arch_key.upper():4s}] SKIP — {len(missing)} models missing '
              f'(e.g. pixel {missing[0]:03d})')
        continue

    print(f'\n{"═"*52}')
    print(f'  {arch_key.upper()} — loading 100 models ...')

    # ── 2. Load ───────────────────────────────────────────────────────────────
    t0 = time.time()
    models = []
    for i in range(N_PIXELS):
        path = os.path.join(
            arch_dir,
            f'model_pixel{i:03d}_size{SIZE}_gen{GEN}_{arch_key}.keras'
        )
        models.append(tf.keras.models.load_model(path, compile=False))
    print(f'  Loaded in {time.time()-t0:.1f}s  |  '
          f'params/model: {models[0].count_params():,}')

    # ── 3. Reshape test set for this architecture ─────────────────────────────
    X_test_reshaped = arch_cfg['reshape_fn'](X_test_4d)

    # ── 4. Predict ────────────────────────────────────────────────────────────
    t0 = time.time()
    n_samples   = X_test_reshaped.shape[0]
    pred_boards = np.zeros((n_samples, N_PIXELS), dtype=np.int8)

    for i, model in enumerate(models):
        probs = model.predict(X_test_reshaped, batch_size=512, verbose=0).flatten()
        pred_boards[:, i] = (probs >= 0.5).astype(np.int8)

    print(f'  Inference done in {time.time()-t0:.1f}s')

    # Free the 100 models immediately to avoid OOM when loading the next arch
    del models
    tf.keras.backend.clear_session()

    # ── 5. Evaluate ───────────────────────────────────────────────────────────
    results = evaluate_full_boards(
        y_true=y_test_full,
        y_pred=pred_boards,
        architecture_name=f'{arch_key.upper()} (size={SIZE}, gen={GEN})',
    )
    all_arch_results[arch_key] = results

print(f'\n✓ Done. Evaluated {len(all_arch_results)} architecture(s).')

In [ ]:
# ─── Combined comparison table ────────────────────────────────────────────────
comparison_all = pd.DataFrame([
    {
        'Architecture':      r['architecture'],
        'Avg Pixel Acc':     r['avg_pixel_acc'],
        'Exact Board Acc':   r['exact_board_acc'],
        'Exact Boards':      f"{r['n_exact_matches']}/{r['n_samples']}",
        'Precision':         r['precision_alive'],
        'Recall':            r['recall_alive'],
        'F1':                r['f1_alive'],
    }
    for r in all_arch_results.values()
]).set_index('Architecture')

# Highlight the best numeric value in each column
numeric_cols = ['Avg Pixel Acc', 'Exact Board Acc', 'Precision', 'Recall', 'F1']
display(
    comparison_all.style
        .highlight_max(subset=numeric_cols, color='#c6efce')
        .format({c: '{:.4f}' for c in numeric_cols})
)

# ─── Side-by-side per-pixel accuracy heatmaps ────────────────────────────────
n_archs = len(all_arch_results)
if n_archs > 0:
    fig, axes = plt.subplots(1, n_archs, figsize=(5 * n_archs, 4))
    if n_archs == 1:
        axes = [axes]

    for ax, (arch_key, r) in zip(axes, all_arch_results.items()):
        grid = np.array(r['pixel_accs_per_pixel']).reshape(SIZE, SIZE)
        im = ax.imshow(grid, vmin=0.5, vmax=1.0, cmap='RdYlGn')
        ax.set_title(f'{arch_key.upper()}\nmean={r["avg_pixel_acc"]:.3f}', fontsize=10)
        ax.set_xlabel('Col')
        ax.set_ylabel('Row')
        for row in range(SIZE):
            for col in range(SIZE):
                ax.text(col, row, f'{grid[row, col]:.2f}',
                        ha='center', va='center', fontsize=5.5, color='black')

    fig.colorbar(im, ax=axes, label='Per-pixel accuracy', shrink=0.8)
    plt.suptitle(f'Per-pixel Test Accuracy — size={SIZE}, gen={GEN}', y=1.02)
    plt.tight_layout()
    plt.show()